## Harmony Py Library
### Requesting Many Variables

Harmony limits a multipart form POST to 100 parts. harmony-py used to send each variable as its own form part, so requests with about 100 or more variables failed. harmony-py now sends all variables in a single comma-separated `variable` part, the same way it sends `granuleId`.

This notebook submits a request with more than 100 variables and checks that it succeeds. It runs against Harmony UAT and authenticates with Earthdata Login credentials from your `.netrc` file.

In [ ]:
import helper
import sys
sys.path.append('..')

# Install harmony-py requirements.  Not necessary if you ran `pip install harmony-py` in your kernel
helper.install_project_and_dependencies('..')

from harmony import Client, Collection, Environment, Request

#### Create a client for Harmony UAT

In [ ]:
harmony_client = Client(env=Environment.UAT)  # assumes .netrc usage

#### Build a request with more than 100 variables

The test collection only has `red_var`, `green_var`, `blue_var` and `alpha_var`, so the list repeats them, as in the example curl request.

In [ ]:
NUM_VARIABLES = 150

base_variables = ['red_var', 'green_var', 'blue_var', 'alpha_var']
variables = [base_variables[i % len(base_variables)] for i in range(NUM_VARIABLES)]

collection = Collection(id='C1233800302-EEDTEST')
request = Request(
    collection=collection,
    granule_id=['G1233800352-EEDTEST'],
    variables=variables,
)
print(f'Number of variables: {len(request.variables)}')

#### Inspect the multipart form body

The body should have exactly one `variable` part holding a comma-separated list, no matter how many variables were requested.

In [ ]:
prepared = harmony_client._get_prepared_request(request)
body = prepared.body

total_parts = body.count(b'Content-Disposition: form-data')
variable_parts = body.count(b'name="variable"')
print(f'Total form parts: {total_parts}')
print(f'"variable" form parts: {variable_parts}')

#### Submit the request and wait for it to finish

In [ ]:
job_id = harmony_client.submit(request)
print(f'Job ID: {job_id}')

harmony_client.wait_for_processing(job_id, show_progress=True)

In [ ]:
job = harmony_client.result_json(job_id)
print(f"Status: {job['status']}")
print(f"Message: {job['message']}")
print(f"Request: {job['request']}")

#### Submit more parts than allowed fails Harmony validation

This sends the same request with one form part per variable, which is what harmony-py used to do. Harmony should reject it because the form has more than 100 parts.

In [ ]:
submit_url = harmony_client._submit_url(request)
old_style_parts = [
    ('forceAsync', (None, 'true')),
    ('granuleId', (None, 'G1233800352-EEDTEST')),
] + [('variable', (None, v)) for v in variables]

response = harmony_client._session().post(submit_url, files=old_style_parts)
print(f'Form parts sent: {len(old_style_parts)}')
print(f'HTTP status: {response.status_code}')
print(response.text[:1000])